In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import optuna
import os
import sklearn

In [2]:
sklearn.set_config(transform_output="pandas")

In [3]:
# Load data - using train_original.csv to get the native NaNs
train = pd.read_csv('train_original.csv')
test = pd.read_csv('test.csv')

In [4]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [5]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

In [6]:
def apply_mappings(df):
    df_out = df.copy()
    # Map ordinals, fill missing in these specific columns with 'Unknown' mapping (-1) if any
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    
    # Leave gender as category
    df_out['gender'] = df_out['gender'].fillna('Unknown').astype('category')
    
    return df_out

In [7]:
X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

In [8]:
# Feature Engineering
denom_screen = X_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif = X_preprocessed['notifications_per_day'].replace(0, 0.001)
X_preprocessed['social_media_ratio'] = X_preprocessed['social_media_hours'] / denom_screen
X_preprocessed['gaming_ratio'] = X_preprocessed['gaming_hours'] / denom_screen
X_preprocessed['work_study_ratio'] = X_preprocessed['work_study_hours'] / denom_screen
X_preprocessed['app_opens_per_hour'] = X_preprocessed['app_opens_per_day'] / denom_screen
X_preprocessed['notifications_to_opens_ratio'] = X_preprocessed['app_opens_per_day'] / denom_notif
X_preprocessed['sleep_deficit'] = 8.0 - X_preprocessed['sleep_hours']

In [9]:
denom_screen_test = X_test_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif_test = X_test_preprocessed['notifications_per_day'].replace(0, 0.001)
X_test_preprocessed['social_media_ratio'] = X_test_preprocessed['social_media_hours'] / denom_screen_test
X_test_preprocessed['gaming_ratio'] = X_test_preprocessed['gaming_hours'] / denom_screen_test
X_test_preprocessed['work_study_ratio'] = X_test_preprocessed['work_study_hours'] / denom_screen_test
X_test_preprocessed['app_opens_per_hour'] = X_test_preprocessed['app_opens_per_day'] / denom_screen_test
X_test_preprocessed['notifications_to_opens_ratio'] = X_test_preprocessed['app_opens_per_day'] / denom_notif_test
X_test_preprocessed['sleep_deficit'] = 8.0 - X_test_preprocessed['sleep_hours']

In [10]:
# Optuna Hyperparameter Tuning
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'is_unbalance': True,
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    # We use a subset of data if we want to tune faster, but for best results we use the whole train set.
    # To keep the tuning somewhat fast on 690k rows, we'll just run it.
    for train_idx, val_idx in cv.split(X_preprocessed, y):
        X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(scores)

In [11]:
print("Starting Optuna tuning (15 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

[I 2026-08-18 17:07:19,940] A new study created in memory with name: no-name-959f0c31-39f1-482c-ad1a-48bbe57b8a9d


Starting Optuna tuning (15 trials)...


[I 2026-08-18 17:07:52,050] Trial 0 finished with value: 0.951956227174831 and parameters: {'n_estimators': 367, 'learning_rate': 0.019046250316028943, 'num_leaves': 148, 'max_depth': 9, 'min_child_samples': 56, 'colsample_bytree': 0.853926811130836, 'subsample': 0.8510225736729781}. Best is trial 0 with value: 0.951956227174831.


[I 2026-08-18 17:08:11,835] Trial 1 finished with value: 0.9624991879815493 and parameters: {'n_estimators': 565, 'learning_rate': 0.08106080976923874, 'num_leaves': 119, 'max_depth': 6, 'min_child_samples': 58, 'colsample_bytree': 0.7680888578893099, 'subsample': 0.823842033351248}. Best is trial 1 with value: 0.9624991879815493.


[I 2026-08-18 17:08:27,103] Trial 2 finished with value: 0.9620642926988049 and parameters: {'n_estimators': 315, 'learning_rate': 0.05496400988999473, 'num_leaves': 122, 'max_depth': 12, 'min_child_samples': 26, 'colsample_bytree': 0.6384442946315471, 'subsample': 0.6756713752088996}. Best is trial 1 with value: 0.9624991879815493.


...

[I 2026-08-18 17:11:00,226] Trial 10 finished with value: 0.9400242007272724 and parameters: {'n_estimators': 207, 'learning_rate': 0.03303142718231492, 'num_leaves': 31, 'max_depth': 4, 'min_child_samples': 30, 'colsample_bytree': 0.9906830623354821, 'subsample': 0.6004300510502384}. Best is trial 4 with value: 0.9626443508295179.


[I 2026-08-18 17:11:16,149] Trial 11 finished with value: 0.9609352119439724 and parameters: {'n_estimators': 596, 'learning_rate': 0.09821458111388409, 'num_leaves': 67, 'max_depth': 4, 'min_child_samples': 53, 'colsample_bytree': 0.7547190056965012, 'subsample': 0.7667470894165551}. Best is trial 4 with value: 0.9626443508295179.


[I 2026-08-18 17:11:40,569] Trial 12 finished with value: 0.9588183615361293 and parameters: {'n_estimators': 535, 'learning_rate': 0.040252088137755354, 'num_leaves': 67, 'max_depth': 6, 'min_child_samples': 64, 'colsample_bytree': 0.6991101135078839, 'subsample': 0.6813817963492779}. Best is trial 4 with value: 0.9626443508295179.


[I 2026-08-18 17:12:08,774] Trial 13 finished with value: 0.9630096587656628 and parameters: {'n_estimators': 520, 'learning_rate': 0.09350577087889028, 'num_leaves': 77, 'max_depth': 8, 'min_child_samples': 48, 'colsample_bytree': 0.7284194633822374, 'subsample': 0.8118559487010616}. Best is trial 13 with value: 0.9630096587656628.


[I 2026-08-18 17:12:26,405] Trial 14 finished with value: 0.9630437217465836 and parameters: {'n_estimators': 512, 'learning_rate': 0.0979442685903844, 'num_leaves': 62, 'max_depth': 9, 'min_child_samples': 41, 'colsample_bytree': 0.7194648477823522, 'subsample': 0.6178470335405051}. Best is trial 14 with value: 0.9630437217465836.


In [12]:
print(f"Best ROC-AUC Score: {study.best_value}")
print(f"Best Params: {study.best_params}")

Best ROC-AUC Score: 0.9630437217465836
Best Params: {'n_estimators': 512, 'learning_rate': 0.0979442685903844, 'num_leaves': 62, 'max_depth': 9, 'min_child_samples': 41, 'colsample_bytree': 0.7194648477823522, 'subsample': 0.6178470335405051}


In [13]:
print("Training final model with best params on ALL data...")
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1
best_params['is_unbalance'] = True
best_params['n_jobs'] = -1

Training final model with best params on ALL data...


In [14]:
final_model = LGBMClassifier(**best_params)
final_model.fit(X_preprocessed, y)

,num_leaves,62
,max_depth,9
,learning_rate,0.0979442685903844
,n_estimators,512
,min_child_samples,41
,subsample,0.6178470335405051
,colsample_bytree,0.7194648477823522
,random_state,42
,n_jobs,-1
,verbose,-1
,is_unbalance,True


In [15]:
print("Predicting final results...")
final_preds = final_model.predict_proba(X_test_preprocessed)[:, 1]

Predicting final results...


In [16]:
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('submissions/native_nan_fixed_optuna.csv', index=False)
print("Submission saved to submissions/native_nan_fixed_optuna.csv")

Submission saved to submissions/native_nan_fixed_optuna.csv
